## Подготовка данных

In [1]:
import numpy as np
import pandas as pd

# СГЕНЕРИРУЙТЕ СВОЙ SEED
MY_SEED = np.random.randint(5000000, size=1)[0]
np.random.seed(MY_SEED)

# СГЕНЕРИРУЙТЕ ЧИСЛО УЧАСТНИКОВ ЭКСПЕРИМЕНТА
n_users = np.random.randint(1000, 10000, size=1)[0]

# Шаг 1: создаём ковариату pre_activity (логинормальное распределение)
pre_activity = np.random.exponential(scale=10, size=n_users).astype(int)

# Шаг 2: НАРУШЕНИЕ 1 — SRM (Sample Ratio Mismatch)
# Вместо 50/50 делаем 40/60
variant = np.random.choice(['control', 'test'], size=n_users, p=[0.4, 0.6])

# Шаг 3: НАРУШЕНИЕ 2 — confounder (рандомизация зависит от pre_activity)
# Для тестовой группы увеличиваем вероятность назначения для активных пользователей
# (это нарушает независимость)
prob_test = 0.4 + 0.3 * (pre_activity / pre_activity.max())
variant2 = np.where(np.random.random(n_users) < prob_test, 'test', 'control')

# Комбинируем оба нарушения (можно взять одно из двух, но для задачи оставим оба)
# Для простоты выберем основное назначение через variant2 (confounder)
variant = variant2

# НАРУШЕНИЕ 3 — дисбаланс guardrail (ошибки чаще в тесте)
# В тесте ошибки встречаются в 2 раза чаще, хотя истинного эффекта нет
error_flag = np.zeros(n_users)
error_flag[variant == 'control'] = np.random.binomial(1, 0.01, sum(variant == 'control'))
error_flag[variant == 'test'] = np.random.binomial(1, 0.02, sum(variant == 'test'))

# Конверсия не зависит от варианта, но зависит от pre_activity
conversion_prob = 0.05 + 0.01 * (pre_activity / pre_activity.max())
converted = np.random.binomial(1, conversion_prob, n_users)

# Шаг 4: Генерация дат (например, за 14 дней, с возможным смещением)
start_date = pd.Timestamp('2026-05-01')
dates = [start_date + pd.Timedelta(days=np.random.randint(0, 14)) for _ in range(n_users)]

df = pd.DataFrame({
    'user_id': range(n_users),
    'variant': variant,
    'converted': converted,
    'error_flag': error_flag,
    'pre_activity': pre_activity,
    'date': dates})

df.to_csv('experiment_data.csv', index=False)
print("Данные сохранены.Идентифицируйте проблемы")


Данные сохранены.Идентифицируйте проблемы


# Решение

In [8]:
import pandas as pd
import scipy

## Задание 1
Загружает experiment_data.csv и выводит базовую информацию (размеры групп, среднюю конверсию, частоту ошибок).

In [6]:
df = pd.read_csv('experiment_data.csv', parse_dates=True)

In [7]:
print('Общий размер выборки', len(df))

print('Количество групп', len(df['variant'].value_counts(normalize=True)))

test_group = df[df['variant'] == 'test']
control_group = df[df['variant'] == 'control']

print('Размер тестовой группы', len(test_group))
print('Размер контрольной группы', len(control_group))

print(f'Средняя конверсия: {df['converted'].mean():.3f}')
print(f'Частота ошибок: {((df['error_flag'] == 1.0).sum() / len(df) * 100):.2f}%')

Общий размер выборки 2591
Количество групп 2
Размер тестовой группы 1121
Размер контрольной группы 1470
Средняя конверсия: 0.047
Частота ошибок: 1.20%


## Задание 2
Проверяет SRM (Sample Ratio Mismatch), например, с помощью хи-квадрат теста.
- Ожидаемое соотношение 50/50.
- Если p-value < 0.05, вывести предупреждение «SRM обнаружен».

In [22]:
print(
    'Проверка соотношения 50/50, если False присутствует большое расхождение -',
    0.95 < (len(test_group) / len(control_group)) < 1.05
)

observed = [len(control_group), len(test_group)]
total_users = sum(observed)
expected = [total_users / 2, total_users / 2]

chi2_stat, p_val = scipy.stats.chisquare(f_obs=observed, f_exp=expected)

print(f"p-value: {p_val:.5f}")
if p_val < 0.05:
    print("SRM обнаружен")
else:
    print("SRM не обнаружен")

Проверка соотношения 50/50, если False присутствует большое расхождение - False
p-value: 0.00000
SRM обнаружен


## Задание 3
Проверяет баланс ковариаты pre_activity между группами.
-    Используйте t-test (или U-тест, если распределение не нормальное).
-    Если различия статистически значимы (p-value < 0.05), вывести предупреждение «Дисбаланс по pre_activity — возможен confounder».

In [23]:
_, p_norm_control = scipy.stats.shapiro(control_group['pre_activity'])
_, p_norm_test = scipy.stats.shapiro(test_group['pre_activity'])

print(f"p-value теста Шапиро-Уилка (Control): {p_norm_control:.5f}")
print(f"p-value теста Шапиро-Уилка (Test): {p_norm_test:.5f}")

is_normal = (p_norm_control >= 0.05) and (p_norm_test >= 0.05)

if is_normal:
    print("Распределения нормальные. Используем t-test...")
    stat, p_val = scipy.stats.ttest_ind(control_group['pre_activity'], test_group['pre_activity'], equal_var=False)
    test_name = "t-test"
else:
    print("Распределения ненормальные. Используем U-test Манна-Уитни...")
    stat, p_val = scipy.stats.mannwhitneyu(control_group['pre_activity'], test_group['pre_activity'], alternative='two-sided')
    test_name = "U-test"

print(f"p-value ({test_name}): {p_val:.5f}")
if p_val < 0.05:
    print("Дисбаланс по pre_activity — возможен confounder")
else:
    print("Дисбаланс по pre_activity не обнаружен")

p-value теста Шапиро-Уилка (Control): 0.00000
p-value теста Шапиро-Уилка (Test): 0.00000
Распределения ненормальные. Используем U-test Манна-Уитни...
p-value (U-test): 0.00184
Дисбаланс по pre_activity — возможен confounder


## Задание 4
Анализирует guardrail-метрику error_flag.
-    Посчитать частоту ошибок в контрольной и тестовой групп

In [24]:
from statsmodels.stats.proportion import proportions_ztest

# Считаем количество ошибок и общий размер групп
errors_control = control_group['error_flag'].sum()
errors_test = test_group['error_flag'].sum()

n_control = len(control_group)
n_test = len(test_group)

rate_control = errors_control / n_control
rate_test = errors_test / n_test

print(f"Частота ошибок в Control: {rate_control * 100:.2f}%")
print(f"Частота ошибок в Test: {rate_test * 100:.2f}%")

# Z-тест для двух долей
stat, p_val = proportions_ztest([errors_control, errors_test], [n_control, n_test])

print(f"p-value (Z-test): {p_val:.5f}")
if p_val < 0.05:
    print("Обнаружено значимое изменение частоты ошибок (guardrail-метрика нарушена)!")
else:
    print("Значимых изменений в частоте ошибок не обнаружено")

Частота ошибок в Control: 0.61%
Частота ошибок в Test: 1.96%
p-value (Z-test): 0.00174
Обнаружено значимое изменение частоты ошибок (guardrail-метрика нарушена)!


## Задание 5
Принимает решение о валидности эксперимента на основе пунктов 2–4:

Если есть хотя бы одно предупреждение → эксперимент не валиден, вывод: «Остановить тест, исправить ошибки, перезапустить».
Если всё в порядке → провести стандартный анализ первичной метрики (конверсии) и сделать вывод о значимости.

In [25]:
# Определяем флаги проблем на основе рассчитанных p-value
srm_warning = (scipy.stats.chisquare([len(control_group), len(test_group)], [total_users/2, total_users/2]).pvalue < 0.05)

_, p_val_mw = scipy.stats.mannwhitneyu(control_group['pre_activity'], test_group['pre_activity'], alternative='two-sided')
confounder_warning = (p_val_mw < 0.05)

_, p_val_z = proportions_ztest([errors_control, errors_test], [n_control, n_test])
guardrail_warning = (p_val_z < 0.05)

# Принимаем решение о валидности
if srm_warning or confounder_warning or guardrail_warning:
    print("Решение: Остановить тест, исправить ошибки, перезапустить")
else:
    print("Эксперимент валиден. Проводим стандартный анализ конверсии...")
    # Тест на равенство конверсий
    conv_control = control_group['converted'].sum()
    conv_test = test_group['converted'].sum()
    stat_conv, p_val_conv = proportions_ztest([conv_control, conv_test], [n_control, n_test])
    print(f"p-value (конверсия): {p_val_conv:.5f}")
    if p_val_conv < 0.05:
        print("Результаты стат. значимы! Есть эффект от изменений.")
    else:
        print("Статистически значимых различий в конверсиях не обнаружено.")

Решение: Остановить тест, исправить ошибки, перезапустить


## Задание 6
Бонус (необязательно, но желательно):

Предложите метод исправления одной из проблем (например, взвешивание наблюдений для коррекции дисбаланса по pre_activity или удаление несостоятельных данных).
Продемонстрируйте, как изменится вывод после исправления (например, отфильтровав данные до сбалансированных по ковариате).